In [ ]:
from pathlib import Path

import pandas as pd

# 1. On définit où sont les fichiers
processed_dir = Path("data/processed")
f_interactions = processed_dir / "interactions.parquet"
f_items = processed_dir / "items.parquet"
f_tags = processed_dir / "item_tags.parquet"  # ou tags.parquet selon ce qu'il a choisi
f_text = processed_dir / "item_text.parquet"

print("🔍 DÉBUT DU CONTRÔLE QUALITÉ\n")

# --- DoD 1 : Présence des fichiers ---
print("1. Vérification de l'existence des fichiers :")
fichiers_attendus = [f_interactions, f_items, f_tags, f_text]
tous_presents = True
for f in fichiers_attendus:
    if f.exists():
        print(f"  ✅ {f.name} est bien là.")
    else:
        print(f"  ❌ ALERTE : {f.name} est introuvable !")
        tous_presents = False

# --- DoD 2 : Cohérence référentielle ---
if tous_presents and f_interactions.exists():
    print("\n2. Vérification de la cohérence référentielle :")
    interactions = pd.read_parquet(f_interactions, columns=["item_id"])
    items = pd.read_parquet(f_items, columns=["item_id"])

    # On compare les ensembles (set)
    items_in_interactions = set(interactions["item_id"])
    items_in_items = set(items["item_id"])

    # Quels items sont dans interactions mais PAS dans items ?
    missing_items = items_in_interactions - items_in_items

    if len(missing_items) == 0:
        print("  ✅ Parfait ! Tous les items cliqués existent dans la table items.")
    else:
        print(f"  ❌ ALERTE : Il y a {len(missing_items)} items fantômes dans les interactions !")

# --- DoD 3 : Zéro valeurs manquantes dans le texte ---
if f_text.exists():
    print("\n3. Vérification des textes (NaN) :")
    texts = pd.read_parquet(f_text)

    # On compte combien de lignes ont un texte manquant
    nb_nans = texts["text"].isna().sum()
    nb_vides = (texts["text"] == "").sum()

    if nb_nans == 0 and nb_vides == 0:
        print("  ✅ Super ! Aucun NaN et aucune chaîne de caractères vide dans les textes.")
    else:
        print(f"  ❌ ALERTE : Il y a {nb_nans} NaN et {nb_vides} textes vides !")

print("\n🏁 FIN DU CONTRÔLE.")

🔍 DÉBUT DU CONTRÔLE QUALITÉ

1. Vérification de l'existence des fichiers :
  ✅ interactions.parquet est bien là.
  ✅ items.parquet est bien là.
  ✅ item_tags.parquet est bien là.
  ✅ item_text.parquet est bien là.

2. Vérification de la cohérence référentielle :
  ✅ Parfait ! Tous les films cliqués existent dans la table items.

3. Vérification des textes (NaN) :
  ✅ Super ! Aucun NaN et aucune chaîne de caractères vide dans les textes.

🏁 FIN DU CONTRÔLE.
